In [ ]:
%load_ext autoreload
%autoreload 2
from hypnose_behavior.visualization.choice import plot_choice_history
from hypnose_behavior.visualization.rewards import (
    plot_cumulative_rewards,
    plot_cumulative_rewards_by_trial,
)
from hypnose_behavior.io.save import use_style
from hypnose_behavior.modelling.ab_learning.data import load_ab_data
from hypnose_behavior.modelling.ab_learning.diagnostics import choice_persistence, lose_shift_summary
from hypnose_behavior.modelling.ab_learning.log_regression import (
    fit_session_models,
    gain_decomposition,
    model_comparison,
    overnight_share,
    shared_slope,
)
from hypnose_behavior.visualization.modelling.ab_learning.log_regression import (
    plot_gain_decomposition,
)
from hypnose_behavior.visualization.modelling.ab_learning.diagnostics import (
    plot_failed_attempt_accuracy,
    plot_initiation_rate,
    plot_lose_shift,
)
import pandas as pd

%matplotlib widget

use_style("presentation")

# Visualize a sessions choice history

In [ ]:
# Plots all choices for one or more sessions
choice_plots = plot_choice_history(
    subjid=45, 
    dates=[20260107], 
    fa_types=["FA_time_in", "FA_time_out"],
    save=False, 
    show_legend=True,
    title=False,
    lw_scale=3,
    xlim=(0,8500),
    marker_scale=3,)

# Subjids

Define sets of subjids to be used in downstream plots (e.g., cohorts with their dates). 

In [ ]:
subjids={53: (20260506, 20260518), 
         55: (20260512, 20260529),
         56: (20260506, 20260529),
         57: (20260526, 20260529),
         58: (20260424, 20260506),
         59: (20260424, 20260506)}
subjids_2={45: (20251217, 20260109),
           46: (20251226, 20260106),
           47: (20251218, 20260105),
           48: (20251218, 20260109),
           49: (20260113, 20260128),
           50: (20260112, 20260120),
           51: (20260114, 20260116),
           52: (20260109, 20260119)}

poster={46: (20251226, 20260106),
        48: (20251218, 20260109),
        51: (20260114, 20260116),
        55: (20260430, 20260529),
        58: (20260423, 20260506),
        59: (20260423, 20260506)}

In [ ]:
# A/B learning cohort: odour-discrimination sessions from stage 2 on.
# Runs of any other stage are dropped by load_ab_data regardless of the selection.
ab_cohort = {60: {"ses_range": (2, 16)},
             61: {"ses_range": (2, 16)},
             62: {"ses_range": (2, 16)},
             63: {"ses_range": (2, 10)},
             64: {"ses_range": (2, 14)},
             65: {"ses_range": (2, 11)},
             66: {"ses_range": (2, 10)}}

# Cumulative rewards over time (1) or trial id (2)

In [ ]:
# Plot cumulative rewards for multiple sessions and subjects. Can be split by days or consecutive. Can use dictionary to define specific dates for each subject. 
plot_cumulative_rewards(subjids=ab_cohort, dates=None, split_days=False, save=True, show_gap_shading=False, show_session_boundaries=False, show_title=False, show_legend=True)

In [ ]:
# Cumulative rewards vs CONTINUOUS TRIAL INDEX (global_trial_id, concatenated across sessions):

plot_cumulative_rewards_by_trial(subjids=subjids, dates=None, save=False)

# Diagnostics: is completed-trials-only defensible?

Loads trials and failed initiation attempts once; every plot below reuses `ab`.

In [ ]:
ab = load_ab_data(ab_cohort)
ab["sessions"]

## D1: initiation rate

Initiated trials / (trials + failed attempts with a poke), per session.

In [ ]:
figs_d1 = plot_initiation_rate(data=ab, save=True)

## D2: accuracy after failed attempts

Port choice after a failed attempt, scored against that attempt's odor, next to completed-trial accuracy. Zero-poke attempts (poke ended before the valve opened) are shown separately.

In [ ]:
figs_d2 = plot_failed_attempt_accuracy(data=ab, save=True)

## D3: lose-shift

Completed trials split by the failed attempt right before them. Elimination would show as `wrong port` above `none`.

In [ ]:
figs_d3 = plot_lose_shift(data=ab, save=True)
lose_shift_summary(ab, by=())

## D4: choice persistence across attempts

For trials with ≥2 (a.) or ≥ 1 (b.) port-visiting failed attempts: agreement between consecutive attempts, and between the last attempt and the trial's choice, against the chance agreement their accuracy implies (p₁p₂ + (1−p₁)(1−p₂), with p per animal, session and odor).

In [ ]:
d4 = choice_persistence(ab)
display(d4["distribution"])
d4["summary"]

# 0.6 Logistic regression: within-session vs overnight

Three nested models per animal, on the completed-trial rows:

| | form | says |
|---|---|---|
| `M_a` | `logit p = a_s` | every gain falls between sessions |
| `M_b` | `logit p = a_s + b·x` | one within-session slope, shared by sessions |
| `M_c` | `logit p = a_s + b_s·x` | a within-session slope per session |

`x` is a trial's position in its session, 0 at the first and 1 at the last.
**The hypothesis test is `M_a vs M_c`** — `M_b` averages an early positive slope against
a late negative one and can come back flat while both are real.

In [ ]:
# Sessions under `min_trials` are dropped whole: too few trials to identify a
# within-session slope, and a short session that is all-correct separates the fit.
fits = fit_session_models(ab, mode="completed")
pd.concat(f["sessions"] for f in fits.values()).query("not kept")

In [ ]:
model_comparison(fits)

In [ ]:
# M_b's single slope per animal: the descriptive average, not the test.
shared_slope(fits)

## Decomposition

From `M_c`, the change in log-odds splits exactly into a within-session and an overnight
part, `W_s = b_s` and `O_s = a_{s+1} − a_s − b_s`, which telescope to the total.
Each is a contrast of `M_c`, so each carries an exact standard error.

`overnight_share` is `sum(O) / total`. It only reads as a percentage while the two
components point the same way — `mixed_signs` marks the animals where they do not, and
`share_of_movement` (`sum|O| / (sum|W| + sum|O|)`) is what stays interpretable there.

In [ ]:
overnight_share(fits)

In [ ]:
# `gap_days` and `sessions_skipped` say how far an overnight boundary actually reaches:
# a dropped session makes one boundary span more than a single night.
gain_decomposition(fits)

In [ ]:
figs_0_6 = plot_gain_decomposition(fits=fits, save=True)

## Robustness: every choice attempt

The same models over the attempt-level rows (plan 0.8) — completed trials plus failed
attempts that ended in a port visit. `x` is renormalized over that axis, so the two modes
never mix.

In [ ]:
fits_attempts = fit_session_models(ab, mode="attempts")
model_comparison(fits_attempts)

In [ ]:
overnight_share(fits_attempts)

In [ ]:
figs_0_6_attempts = plot_gain_decomposition(fits=fits_attempts, save=False)